In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import torch
import torch.optim as optim
import torch.nn as nn
from torchviz import make_dot

# Gradient Descent

In [2]:
true_b = 1
true_w = 2
N = 100

np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_w * x + epsilon

In [3]:
idx = np.arange(N)
np.random.shuffle(idx)

train_idx = idx[:int(N* .8)]
val_idx = idx[int(N* .8):]

x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

In [4]:
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)
print(b, w)

[0.49671415] [-0.1382643]


In [5]:
yhat = b + w * x_train

In [6]:
error = (yhat - y_train)
loss = (error ** 2).mean()
print(loss)

2.7421577700550976


In [7]:
b_grad = 2 * error.mean()
w_grad = 2 * (x_train * error).mean()
print(b_grad, w_grad)

-3.044811379650508 -1.8337537171510832


In [8]:
lr = 0.1
print(b, w)

b = b - lr * b_grad
w = w - lr * w_grad

print(b, w)

[0.49671415] [-0.1382643]
[0.80119529] [0.04511107]


# Linear Regression in Numpy

In [9]:
# step 0
np.random.seed(42)
b = np.random.randn(1)
w = np.random.randn(1)
print(b, w)

# sets lr
lr = 0.1
n_epochs = 1000
for epoch in range(n_epochs):
    # step 1
    yhat = b + w * x_train
    
    # step 2
    error = (yhat - y_train)
    loss = (error ** 2).mean()

    # step 3
    b_grad = 2 * error.mean()
    w_grad = 2 * (x_train * error).mean()

    # step 4
    b = b - lr * b_grad
    w = w - lr * w_grad

print(b, w)

[0.49671415] [-0.1382643]
[1.02354094] [1.96896411]


In [10]:
# use sklearn
linr = LinearRegression()
linr.fit(x_train, y_train)
print(linr.intercept_, linr.coef_[0])

[1.02354075] [1.96896447]


# PyTorch

In [11]:
scalar = torch.tensor(3.14159)
vector = torch.tensor([1, 2, 3])
matrix = torch.ones((2, 3), dtype=torch.float)
tensor = torch.randn((2, 3, 4), dtype=torch.float)

print(scalar)
print(vector)
print(matrix)
print(tensor)

tensor(3.1416)
tensor([1, 2, 3])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[[-1.9617, -0.4524,  1.4424,  1.8888],
         [-0.4408,  0.2906, -1.1640,  0.9450],
         [-0.5067,  0.6738,  0.2745, -0.3546]],

        [[ 1.0404,  0.0066,  0.3450, -1.5726],
         [ 0.3297, -1.3961,  0.9009,  0.6638],
         [ 0.5924, -2.1995, -0.1593,  1.0245]]])


In [12]:
print(tensor.size(), tensor.shape)

torch.Size([2, 3, 4]) torch.Size([2, 3, 4])


In [13]:
print(scalar.size(), scalar.shape)

torch.Size([]) torch.Size([])


In [14]:
same_matrix = matrix.view(1, 6)
same_matrix[0, 1] = 2
print(matrix)
print(same_matrix)

tensor([[1., 2., 1.],
        [1., 1., 1.]])
tensor([[1., 2., 1., 1., 1., 1.]])


In [15]:
different_matrix = matrix.new_tensor(matrix.view(1, 6))
different_matrix[0, 1] = 3
print(matrix)
print(different_matrix)

tensor([[1., 2., 1.],
        [1., 1., 1.]])
tensor([[1., 3., 1., 1., 1., 1.]])


C:\Users\duy.ho\AppData\Local\Temp\ipykernel_6580\2171686589.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  different_matrix = matrix.new_tensor(matrix.view(1, 6))


In [16]:
# clone method
another_matrix = matrix.view(1, 6).clone().detach()
another_matrix[0, 1] = 4
print(matrix)
print(another_matrix)

tensor([[1., 2., 1.],
        [1., 1., 1.]])
tensor([[1., 4., 1., 1., 1., 1.]])


In [17]:
x_train_tensor = torch.as_tensor(x_train)
x_train.dtype, x_train_tensor.dtype

(dtype('float64'), torch.float64)

In [18]:
float_tensor = x_train_tensor.float()
float_tensor.dtype

torch.float32

In [20]:
dummy_array = np.array([1, 2, 3])
dummy_tensor = torch.as_tensor(dummy_array)
# modifies the numpy array
dummy_array[1] = 0
dummy_tensor

tensor([1, 0, 3])

In [21]:
dummy_tensor.numpy()

array([1, 0, 3])

In [22]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [24]:
n_cudas = torch.cuda.device_count()
for i in range(n_cudas):
    print(torch.cuda.get_device_name(i))

In [25]:
gpu_tensor = torch.as_tensor(x_train).to(device)
gpu_tensor[0]

tensor([0.7713], dtype=torch.float64)

In [27]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# data was is in Numpy arrays, but we need transfrom them in to pyorch tensors and then send them to the choosen device
x_train_tensor = torch.as_tensor(x_train).float().to(device)
y_train_tensor = torch.as_tensor(y_train).float().to(device)

In [29]:
# see the difference - notice that .tyoe() is more useful since it also tell us where the tensor is (device)
print(type(x_train), type(x_train_tensor), x_train_tensor.type())

<class 'numpy.ndarray'> <class 'torch.Tensor'> torch.FloatTensor


In [31]:
back_to_numpy = x_train_tensor.cpu().numpy()

In [33]:
# first
# apply gradient descent on these parameters we need to set requires_grad = true
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float)
w = torch.randn(1, requires_grad=True, dtype=torch.float)
print(b, w)

tensor([0.3367], requires_grad=True) tensor([0.1288], requires_grad=True)


In [34]:
# second
# use GPU
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, dtype=torch.float).to(device)
w = torch.randn(1, requires_grad=True, dtype=torch.float).to(device)
print(b, w)

tensor([0.3367], requires_grad=True) tensor([0.1288], requires_grad=True)


In [35]:
# thrid
torch.manual_seed(42)
b = torch.randn(1, dtype=torch.float).to(device)
w = torch.randn(1, dtype=torch.float).to(device)

# set requiring gradients ....
b.requires_grad_()
w.requires_grad_()
print(b, w)

tensor([0.3367], requires_grad=True) tensor([0.1288], requires_grad=True)


In [36]:
# final
# specify the device at the moment of creation

# step 0 - initializes parameters "b" and "w" randomly
torch.manual_seed(42)
b = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
w = torch.randn(1, requires_grad=True, \
                dtype=torch.float, device=device)
print(b, w)

tensor([0.3367], requires_grad=True) tensor([0.1288], requires_grad=True)
